In [1]:
import pandas as pd
import numpy as np
import requests
import json
from time import sleep

In [2]:
compare_plays = pd.read_csv('tracking_data_play_2025121409.csv') 
compare_plays

,game_id,gsis_play_id,nfl_id,display_name,route_break_frames,route_break_depth
0,2025121409,3688,52541.0,Colby Parkinson,['2025121409-3688-179'],0.0
1,2025121409,3688,53486.0,Tutu Atwell,['2025121409-3688-212'],19.0
2,2025121409,3688,58251.0,Terrance Ferguson,['2025121409-3688-205'],12.0
3,2025121409,3380,56042.0,Puka Nacua,['2025121409-3380-199'],1.0
4,2025121409,3380,58251.0,Terrance Ferguson,['2025121409-3380-212'],3.0
...,...,...,...,...,...,...
281,2025121409,78,58275.0,Isaac TeSlaa,['2025121409-78-223'],15.0
282,2025121409,4119,53541.0,Amon-Ra St. Brown,['2025121409-4119-183'],2.0
283,2025121409,4119,54477.0,Jameson Williams,['2025121409-4119-183'],3.0
284,2025121409,988,53541.0,Amon-Ra St. Brown,['2025121409-988-201'],1.0


In [3]:
# Define parameters to access the identity API
USER = "nicholas.esposito"
API_KEY = "31bc0e9f40ca6fcf79f898de4c02b17deeb2be8cb91a26b7037198ef7bca00a2"

def pull_plays(game_id):

    url = f"https://api.profootballfocus.com//v1/premium/nfl/2025/team_main"
    print(url)
    # Get JWT
    auth_resp = requests.post(
        "https://api.profootballfocus.com/auth/login",
        headers={"x-api-key": API_KEY}
    )
    jwt = auth_resp.json()["jwt"]
    print(auth_resp)

    # Use JWT for pass rush
    headers = {"Authorization": f"Bearer {jwt}"}
    # print(headers)
    resp = requests.get(
        url,
        headers=headers,
        params={"page_size": 1}
    )

    res_json = resp.json()
    print(resp)
    print(res_json)

    pff_route_depth = pd.json_normalize(res_json['rec_route_depth'], sep = '-')

    # Filter to only what we need
    pff_route_depth_filtered = pff_route_depth

    # Access player API using JWT - we need nfl id from here (Note: gsis_id = NGS nfl_id)
    headers = {"Authorization": f"Bearer {jwt}"}
    resp = requests.get(
        url,
        headers=headers,
        params={"page_size": 1}
    )

    res_json = resp.json()
    print(res_json)

    pff_players = pd.json_normalize(res_json['players'], sep = '-')

    # Filter to only what we need
    pff_players_filtered = pff_players[['gsis_id', 'id']]

    # Now, join pff_players_filtered onto pff_route_depth_filtered (id = player_id) so the resulting dataframe has 'gsis_game_id', 'gsis_play_id', 'player_id', 'stance', and 'gsis_id'
    pff_merged = pff_route_depth_filtered.merge(
        pff_players_filtered,
        left_on = "player_id",
        right_on = "id",
        how = "left"
    )

    return pff_merged

In [ ]:
pull_plays('2025121409')

https://api.profootballfocus.com//v1/premium/nfl/2025/team_main
<Response [200]>
<Response [200]>
